# CQT Baseline Trainer

L'exécution de ce notebook a pour prérequis :
- Le téléchargement du dataset `GuitarSet`,
- le démarrage de l'infrastructure docker,
- l'ingestion du dataset `GuitarSet`,
- le prétraitement du dataste `GuitarSet`.

Pour télécharger le dataset `GuitarSet`, utilisez la commande :
```bash
uv run ./audio_midi/main.py --download_datasets --no_idmt_smt_guitar
```

Pour démarrer l'infrastructure docker, utilisez la commande :
```bash
docker-compose up -d
```

Pour lancer la pipeline d'ingestion, utilisez la commande :
```bash
uv run ./audio_midi/main.py --ingest_guitar_set
```

Pour lancer la pipeline de prétraitement, utilisez la commande :
```bash
uv run ./audio_midi/main.py --preprocess_datasets --no_idmt_smt_guitar
```

## Imports

In [1]:
import sys
from pathlib import Path

APP_DIR = Path.cwd().parent
sys.path.append(APP_DIR.as_posix())

In [2]:
# Chemins
OUTPUT_DIR = APP_DIR / "output"
ARTIFACT_DIR = OUTPUT_DIR / "cqt_baseline"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Imports graphiques
import matplotlib.pyplot as plt
import seaborn as sns

# Accessibilité : Daltonisme, Dyslexie, Confort Visuel
sns.set_theme(
    style="whitegrid",
    palette="colorblind",
    context="notebook",
)

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.family": "Arial",
        "font.size": 12,
        "axes.titlesize": 15,
        "axes.titleweight": "bold",
        "axes.labelsize": 13,
        "axes.labelweight": "medium",
        "axes.edgecolor": "black",
        "axes.linewidth": 1.2,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "lines.linewidth": 2.2,
        "lines.markersize": 7,
        "legend.fontsize": 11,
        "legend.frameon": True,
        "legend.framealpha": 0.95,
        "grid.linestyle": ":",
        "grid.linewidth": 0.8,
        "grid.alpha": 0.6,
    }
)

COLORBLIND_PALETTE = sns.color_palette("colorblind")

In [ ]:
import os
import json
import warnings
import logging
from datetime import datetime
from time import perf_counter
import functools

import numpy as np
import pandas as pd

from scipy.stats import randint, loguniform

from sklearn.base import ClassifierMixin
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    hamming_loss,
    accuracy_score,
)
from sklearn.model_selection import learning_curve, cross_val_score, TimeSeriesSplit, RandomizedSearchCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA

import mlflow
from mlflow.tracking import MlflowClient

from src.pipelines import DatasetBuilderPipeline
from settings.dataset_builder_pipeline_settings import DatasetBuilderPipelineSettings
from settings import MLFLOW_SETTINGS

warnings.filterwarnings("ignore")

RANDOM_STATE = 73
np.random.seed(RANDOM_STATE)

In [5]:
os.environ["AWS_ACCESS_KEY_ID"] = MLFLOW_SETTINGS.aws_access_key_id
os.environ["AWS_SECRET_ACCESS_KEY"] = MLFLOW_SETTINGS.aws_secret_access_key
os.environ["MLFLOW_S3_ENDPOINT_URL"] = MLFLOW_SETTINGS.s3_endpoint_url
os.environ["AWS_REGION"] = MLFLOW_SETTINGS.aws_region

## Configuration MLflow

In [6]:
def get_or_restore_experiment(experiment_name: str) -> str:
    client = MlflowClient()

    experiment = client.get_experiment_by_name(experiment_name)

    if experiment is None:
        return client.create_experiment(experiment_name)

    if experiment.lifecycle_stage == "deleted":
        client.restore_experiment(experiment.experiment_id)

    return experiment

In [7]:
MLFLOW_EXPERIMENT_NAME = "cqt_baseline"

mlflow.set_tracking_uri(MLFLOW_SETTINGS.tracking_uri)

experiment = get_or_restore_experiment(MLFLOW_EXPERIMENT_NAME)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print("MLflow tracking URI :", mlflow.get_tracking_uri())
print("Experiment ID       :", experiment.experiment_id)
print("Experiment name     :", experiment.name)
print("Lifecycle stage     :", experiment.lifecycle_stage)

MLflow tracking URI : http://localhost:5000
Experiment ID       : 1
Experiment name     : cqt_baseline
Lifecycle stage     : active


## Chargement des données

In [8]:
settings_standard = DatasetBuilderPipelineSettings()
dataset_builder_pipeline = DatasetBuilderPipeline(
    logging.getLogger(), settings=settings_standard
)

train_dataset, validation_dataset, test_dataset = dataset_builder_pipeline.run()

In [9]:
train_features, train_target = train_dataset
X_train = train_features.values
y_train = train_target.values

validation_features, validation_target = validation_dataset
X_validation = validation_features.values
y_validation = validation_target.values

test_features, test_target = test_dataset
X_test = test_features.values
y_test = test_target.values

feature_names = train_features.columns.to_list()
target_names = train_target.columns.to_list()

print(
    f"Dimension jeu d'entrainement : features={X_train.shape}, target={y_train.shape}"
)
print(
    f"Dimension jeu de validation  : features={X_validation.shape}, target={y_validation.shape}"
)
print(f"Dimension jeu de test        : features={X_test.shape}, target={y_test.shape}")
print()

print("Noms des features :", feature_names)
print("Noms des targets  :", target_names)
print()

Dimension jeu d'entrainement : features=(342529, 84), target=(342529, 49)
Dimension jeu de validation  : features=(38591, 84), target=(38591, 49)
Dimension jeu de test        : features=(91440, 84), target=(91440, 49)

Noms des features : ['cqt_0', 'cqt_1', 'cqt_2', 'cqt_3', 'cqt_4', 'cqt_5', 'cqt_6', 'cqt_7', 'cqt_8', 'cqt_9', 'cqt_10', 'cqt_11', 'cqt_12', 'cqt_13', 'cqt_14', 'cqt_15', 'cqt_16', 'cqt_17', 'cqt_18', 'cqt_19', 'cqt_20', 'cqt_21', 'cqt_22', 'cqt_23', 'cqt_24', 'cqt_25', 'cqt_26', 'cqt_27', 'cqt_28', 'cqt_29', 'cqt_30', 'cqt_31', 'cqt_32', 'cqt_33', 'cqt_34', 'cqt_35', 'cqt_36', 'cqt_37', 'cqt_38', 'cqt_39', 'cqt_40', 'cqt_41', 'cqt_42', 'cqt_43', 'cqt_44', 'cqt_45', 'cqt_46', 'cqt_47', 'cqt_48', 'cqt_49', 'cqt_50', 'cqt_51', 'cqt_52', 'cqt_53', 'cqt_54', 'cqt_55', 'cqt_56', 'cqt_57', 'cqt_58', 'cqt_59', 'cqt_60', 'cqt_61', 'cqt_62', 'cqt_63', 'cqt_64', 'cqt_65', 'cqt_66', 'cqt_67', 'cqt_68', 'cqt_69', 'cqt_70', 'cqt_71', 'cqt_72', 'cqt_73', 'cqt_74', 'cqt_75', 'cqt_76', 

## Définition des modèles

### OneVSRestClassifier + LogisticRegression

In [10]:
def get_ovr_lr_model():
    scaler = StandardScaler()

    model = OneVsRestClassifier(
        LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
        )
    )

    return Pipeline(
        steps=[
            ("scaler", scaler),
            ("model", model),
        ]
    )

### OneVSRestClassifier + LinearSVC

In [11]:
def get_ovr_svm_model():
    scaler = StandardScaler()

    model = OneVsRestClassifier(
        LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
    )

    return Pipeline(
        steps=[
            ("scaler", scaler),
            ("model", model),
        ]
    )

### OneVSRestClassifier + RandomForestClassifier

In [12]:
def get_ovr_rf_model():
    scaler = StandardScaler()

    model = OneVsRestClassifier(
        RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            n_jobs=-1,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
        )
    )

    return Pipeline(
        steps=[
            ("scaler", scaler),
            ("model", model),
        ]
    )

### OneVSRestClassifier + HistGradientBoostingClassifier

In [ ]:
def get_ovr_hgb_model():
    scaler = StandardScaler()

    model = OneVsRestClassifier(
        HistGradientBoostingClassifier(
            learning_rate=0.1,
            max_depth=8,
            max_iter=200,
            random_state=RANDOM_STATE,
        )
    )

    return Pipeline(
        steps=[
            ("scaler", scaler),
            ("model", model),
        ]
    )

### OneVSRestClassifier + SGDClassifier

In [14]:
def get_ovr_sgd_model():
    scaler = StandardScaler()

    model = OneVsRestClassifier(
        SGDClassifier(
            loss="log_loss",
            alpha=1e-4,
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
    )

    return Pipeline(
        steps=[
            ("scaler", scaler),
            ("model", model),
        ]
    )

## Evaluation des modèles

La transcription audio → MIDI est formulée comme un problème de classification multi-label frame-wise :

- chaque ligne correspond à une frame temporelle ;
- chaque colonne correspond à une note MIDI ;
- plusieurs notes peuvent être actives simultanément.

Exemple :

| Frame | C4 | D4 | E4 | F4 |
|---------|----|----|----|----|
| t₁ | 1 | 0 | 1 | 0 |
| t₂ | 0 | 0 | 1 | 1 |

Une erreur peut donc être commise :
- sur une note spécifique (par exemple une note non détectée),
- sur une frame complète (par exemple une frame non parfaitement transcrite)
- sur la structure musicale globale (par exemple une note transcrite discontinuement qui devrait être continue).

Nous utilisons donc plusieurs métriques complémentaires pour capturer tous ces aspects.

### F1-score Micro

**Définition :** Le F1-score est la moyenne harmonique entre la précision et le rappel.
Dans le cas **micro**, tous les labels de toutes les frames sont regroupés avant calcul.

$$
Precision_{micro}
=
\frac{\sum TP}
{\sum TP + \sum FP}
$$

$$
Recall_{micro}
=
\frac{\sum TP}
{\sum TP + \sum FN}
$$

$$
F1_{micro}
=
2 \cdot
\frac{
Precision_{micro}
\cdot
Recall_{micro}
}
{
Precision_{micro}
+
Recall_{micro}
}
$$

**Utilité :** Cette métrique répond à la question : "Quelle est la qualité globale de la transcription ?"
Toutes les prédictions sont considérées ensemble, c'est à dire, toutes les notes, toutes les frames, tous les morceaux.

**Interprétation :**

| Valeur | Interprétation |
| :- | :- |
| 1.0 | transcription parfaite |
| > 0.9 | excellente |
| 0.8 - 0.9 | très bonne |
| 0.7 - 0.8 | correcte |
| < 0.7 | amélioration nécessaire |

Le F1 micro constitue la métrique principale pour comparer plusieurs modèles.

### F1-score Macro

**Définition :** On calcule d'abord un F1-score pour chaque note MIDI $F1_k$, puis on effectue la moyenne :

$$
F1_{macro}
=
\frac{1}{K}
\sum_{k=1}^{K}
F1_k
$$

où $k$ représente le nombre total de notes MIDI modélisées.

**Utilité :** Le F1 micro est dominé par les notes les plus fréquentes.
Le F1 macro donne le même poids à une note très fréquente et à une note très rare.
Il permet donc d'évaluer la capacité du modèle à généraliser sur l'ensemble du registre de la guitare.

**Interprétation :**
Un écart important entre $F1_{micro} \gg F1_{macro}$ indique généralement que les notes fréquentes sont bien reconnues et que les notes rares sont mal reconnues. Ce peut être le signe d'un déséquilibre de classes.

### Precision

**Définition :**

$$
Precision = \frac{TP}{TP + FP}
$$

où TP signifie True Positives et FP signifie False Positives.

**Utilité :** La précision répond à la question : "Quand le modèle prédit une note, a-t-il raison ?". Une faible précision signifie que le modèle ajoute beaucoup de notes inexistantes.

**Interprétation :**
Une précision faible révèle un grand nombre de notes inexistantes et une transcription surchargée.
Une précision élevée indique qu'il y a peu de fausses notes et que la transcription est propre.

### Recall

**Définition :**

$$
Recall = \frac{TP}{TP + FN}
$$

où TP signifie True Positives et FN signifie False Negatives

**Utilité :** Le rappel répond à la question : "Combien de vraies notes le modèle retrouve-t-il ?"

**Interprétation :**
Un recall faible révèle que le modèle oublie des notes et que transcription incomplète.
Un recall élevé montre que davantage de notes sont détectées, parfois au prix de faux positifs supplémentaires.

### Hamming Loss

**Définition :**

$$
HammingLoss = \frac{FP + FN}{N \times K}
$$

avec $N$ le nombre de frames et $K$ le nombre de notes MIDI.

**Utilité :** Cette métrique mesure le taux d'erreur moyen par note et par frame.
Contrairement au F1-score, elle pénalise directement chaque erreur élémentaire.


**Interprétation :**

| Valeur | Signification |
| :- | :- |
| 0 | aucune erreur |
| 0.01 | 1 % d'erreurs |
| 0.05 | 5 % d'erreurs |
| 0.10 | 10 % d'erreurs |

Plus la valeur est faible, meilleur est le modèle.

### Subset Accuracy

**Définition :** Une frame est correcte uniquement si toutes les notes sont correctement prédites.

$$
SubsetAccuracy = \frac{\#\;frames\;parfaites}{\#\;frames}
$$

**Utilité :** Cette métrique est extrêmement stricte.
Elle répond à la question : "Combien de frames sont parfaitement transcrites ?"

**Interprétation :**
Même un très bon modèle obtient souvent une valeur relativement faible.
Cette métrique permet de mesurer la qualité des accords complets.

In [15]:
def compute_ml_metrics(y_true, y_pred):
    return {
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_micro": precision_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "recall_micro": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "subset_accuracy": accuracy_score(y_true, y_pred),
    }

### F1-score par pitch MIDI

**Définition :** Pour chaque note MIDI :

$$
F1_k = 2 \cdot \frac{Precision_k \cdot Recall_k}{Precision_k + Recall_k}
$$

**Utilité :** Le score global peut masquer des difficultés spécifiques.
Certaines notes peuvent être très bien reconnues et d'autres très mal reconnues.

Le F1 par pitch permet d'identifier les zones du manche difficiles, les fréquences mal représentées ou les erreurs de feature engineering.

**Interprétation :** Un graphique F1 par pitch permet de visualiser les notes problématiques, les tendances graves / aigus et les limites du modèle.

In [16]:
def compute_f1_per_pitch(y_true, y_pred, pitch_offset=36):
    scores = []

    for k in range(y_true.shape[1]):
        scores.append(
            {
                "pitch_midi": k + pitch_offset,
                "f1_score": f1_score(
                    y_true[:, k],
                    y_pred[:, k],
                    average="binary",
                    zero_division=0,
                ),
            }
        )

    return pd.DataFrame(scores)

### Pitch Tolerance Accuracy

**Définition :** Une prédiction est considérée correcte si elle est proche de la vraie note :

$$
|Pitch_{pred} - Pitch_{true}| \leq t
$$

où $t$ représente le nombre de demi-tons.

**Utilité :** Une erreur d'un demi-ton est moins grave musicalement qu'une erreur d'une octave.
Le F1-score classique considère pourtant ces deux erreurs comme identiques.
Cette métrique introduit une notion de proximité musicale.

**Interprétation :** Une pitch tolerance élevée indique que le modèle comprend globalement les hauteurs de notes.
Une pitch tolerance faible indique que le modèle commet des erreurs importantes sur les hauteurs de notes.

In [17]:
def pitch_tolerance_accuracy(y_true, y_pred, tolerance=1):
    true_idx = np.where(y_true == 1)
    pred_idx = np.where(y_pred == 1)

    if len(true_idx[0]) == 0:
        return 0.0

    correct = 0

    for i in range(len(true_idx[0])):
        t_frame = true_idx[0][i]
        t_pitch = true_idx[1][i]

        frame_preds = pred_idx[1][pred_idx[0] == t_frame]

        if len(frame_preds) == 0:
            continue

        if np.any(np.abs(frame_preds - t_pitch) <= tolerance):
            correct += 1

    return correct / len(true_idx[0])

### Activation Ratio

**Définition :**
$$
ActivationRatio = \frac{\text{taux d'activation prédit}}{\text{taux d'activation réel}}
$$

**Utilité :** Cette métrique mesure le biais global du modèle.

**Interprétation :**
- $Ratio \approx 1$ : Le modèle produit globalement le bon nombre de notes.
- $Ratio > 1$ : Le modèle sur-prédit, il ajoute trop de notes.
- $Ratio < 1$ : Le modèle sous-prédit, il manque des notes.

In [18]:
def activation_ratio(y_true, y_pred):
    return {
        "true_activation": y_true.mean(),
        "pred_activation": y_pred.mean(),
        "ratio": (y_pred.mean() / (y_true.mean() + 1e-8)),
    }

### Temporal Jitter

**Définition :** Le jitter mesure les variations de prédictions entre frames successives.
Une approximation simple est :

$$
Jitter = mean \left(|y_t - y_{t-1}| \right)
$$

**Utilité :** La transcription frame-wise produit souvent un phénomène appelé *flickering*.
Une note apparaît puis disparaît très rapidement alors qu'elle devrait rester stable.

**Interprétation :**
Un jitter faible indique une transcription stable avec des notes continues.
Un jitter élevé révèle une instabilité temporelle.

In [19]:
def temporal_jitter(y_pred):
    return np.mean(np.abs(np.diff(y_pred, axis=0)))

### Pitch Class Confusion Matrix

**Définition :**
Une note MIDI peut être ramenée à sa classe de hauteur (Pitch Class) : $PitchClass = MIDI \bmod 12$
Les notes séparées d'une ou plusieurs octaves appartiennent donc à la même classe.

**Utilité :** La Pitch Class Confusion Matrix regroupe les notes par nom musical (C, C#, D, D#, E, F, F#, G, G#, A, A#, B).
Elle permet de mettre en évidence des erreurs harmoniques ou tonales.
Deux erreurs peuvent avoir le même impact sur le F1-score, par exemple, prédire E au lieu de F et prédire E au lieu de A#.
Pourtant musicalement, ces erreurs sont très différentes.
La Pitch Class Confusion Matrix permet d'analyser la nature musicale des erreurs plutôt que leur simple quantité.

**Interprétation :**
Une diagonale dominante indique que les classes de hauteur sont correctement reconnues.
Des valeurs importantes hors diagonale indiquent des confusions entre notes voisines, des difficultés dans certaines régions fréquentielles et d'éventuels problèmes liés aux harmoniques de la guitare.

In [20]:
def pitch_class_confusion(y_true, y_pred):
    true_pc = np.where(y_true == 1)[1] % 12
    pred_pc = np.where(y_pred == 1)[1] % 12

    cm = np.zeros((12, 12))

    for t, p in zip(true_pc, pred_pc):
        cm[t, p] += 1

    return cm

### Validation croisée (Cross Validation)

**Principe :** Le modèle est entraîné plusieurs fois sur des sous-ensembles différents du dataset.
On calcule pour chaque entraînement le score F1, puis on calcule la moyenne et l'écart-type des scores F1.

**Utilité :** Un bon score sur un seul split peut être dû au hasard.
La validation croisée permet d'évaluer la robustesse la stabilité, et la capacité de généralisation.

**Interprétation :**
Une moyenne élevée indique de bonnes performances globales.
Un écart-type faible montre un comportement stable.
Un écart-type élevé révèle que le modèle est sensible au split.

In [21]:
def run_cv(model, X, y):
    cv = TimeSeriesSplit(n_splits=5)
    scores = cross_val_score(model, X, y, cv=cv, scoring="f1_micro", n_jobs=-1)

    return {"cv_f1_mean": scores.mean(), "cv_f1_std": scores.std()}

In [22]:
def evaluate(
    model,
    X_test,
    y_test,
    label_names,
    pitch_offset=40,  # /!\ Regarder les settings de la pipeline de prétraitement
):
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)
    else:
        y_score = None

    report = classification_report(
        y_test,
        y_pred,
        target_names=label_names,
        output_dict=True,
        zero_division=0,
    )

    metrics = {}
    metrics.update(compute_ml_metrics(y_test, y_pred))

    metrics["pitch_acc_tol_1"] = pitch_tolerance_accuracy(y_test, y_pred, tolerance=1)
    metrics["pitch_acc_tol_2"] = pitch_tolerance_accuracy(y_test, y_pred, tolerance=2)

    metrics.update(activation_ratio(y_test, y_pred))

    metrics["jitter"] = temporal_jitter(y_pred)

    df_f1_per_pitch = compute_f1_per_pitch(y_test, y_pred, pitch_offset)

    cm = confusion_matrix(y_test.flatten(), y_pred.flatten())

    pitch_class_cm = pitch_class_confusion(y_test, y_pred)

    artifacts = {
        "y_pred": y_pred,
        "y_score": y_score,
        "confusion_matrix": cm,
        "classification_report": report,
        "f1_per_pitch": df_f1_per_pitch,
        "pitch_class_confusion_matrix": pitch_class_cm,
    }

    return metrics, artifacts

In [23]:
def log_confusion_matrix(cm, artifact_file="confusion_matrix.png"):
    cm_percent = cm / cm.sum().sum() * 100

    plt.figure(figsize=(7, 5))

    sns.heatmap(
        cm_percent,
        annot=True,
        fmt=".2f",
        cmap="cividis",
        square=True,
        linewidths=0.6,
        linecolor="white",
        annot_kws={"size": 10},
    )

    plt.title("Matrice de confusion")
    plt.xlabel("Predict label")
    plt.ylabel("True label")

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

In [24]:
def log_f1_per_pitch(df_scores, artifact_file="f1_per_pitch.png"):
    plt.figure(figsize=(10, 4))

    plt.plot(
        df_scores["pitch_midi"],
        df_scores["f1_score"],
    )

    plt.title("F1-score per Pitch")
    plt.xlabel("MIDI Pitch")
    plt.ylabel("F1-score")

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

In [25]:
def log_precision_recall_curve(
    y_true, y_score, artifact_file="precision_recall_curve.png"
):
    precision, recall, _ = precision_recall_curve(
        y_true.flatten(),
        y_score.flatten(),
    )

    plt.figure(figsize=(6, 6))

    plt.plot(recall, precision)

    plt.title("Global Precision-Recall Curve")
    plt.xlabel("Recall")
    plt.ylabel("Precision")

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

### Learning Curve

**Principe :**
On entraîne le modèle avec des fractions croissantes du dataset (10%, 20%, 40%, 60%, 80%, 100%) et on mesure les performances.

**Utilité :** La courbe d'apprentissage permet de répondre à plusieurs questions :
- manque-t-on de données ?
- le modèle sous-apprend-il ?
- le modèle sur-apprend-il ?

**Interprétation :**
- Train élevé + Validation faible : sur-apprentissage.
- Train faible + Validation faible : sous-apprentissage.
- Train et Validation convergent : comportement sain.
- Validation continue à monter : davantage de données pourraient améliorer les performances.

In [26]:
def log_learning_curve(model, X, y, artifact_file="learning_curve.png"):
    train_sizes, train_scores, val_scores = learning_curve(
        model,
        X,
        y,
        cv=5,
        scoring="f1_micro",
        n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 5),
    )

    plt.figure(figsize=(8, 5))

    plt.plot(
        train_sizes,
        train_scores.mean(axis=1),
        label="train",
    )

    plt.plot(
        train_sizes,
        val_scores.mean(axis=1),
        label="validation",
    )

    plt.title("Learning Curve")
    plt.xlabel("Training Samples")
    plt.ylabel("F1 Micro")

    plt.legend()

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

## Expériences

In [ ]:
def run_experiment(model_factory, run_name, tags):

    print("=" * 80)
    print(f"[{datetime.now()}] Starting run: {run_name}")

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id
        print(f"[{datetime.now()}] MLflow run_id: {run_id}")

        mlflow.set_tags(tags)

        print(f"[{datetime.now()}] Building model...")
        model = model_factory()

        print(f"[{datetime.now()}] Logging dataset name and id...")
        mlflow.log_params({"dataset_id": dataset_builder_pipeline.pipeline_metadata["_id"]})
        mlflow.log_params({"dataset_name": settings_standard.output_dataset_name})

        print(f"[{datetime.now()}] Training model...")
        t0 = perf_counter()
        model.fit(
            X_train,
            y_train,
        )
        print(f"[{datetime.now()}] Training completed ({perf_counter() - t0:.1f}s)")

        print(f"[{datetime.now()}] Logging model parameters...")
        mlflow.log_params(model.get_params())

        print(f"[{datetime.now()}] Evaluating on test set...")
        metrics, artifacts = evaluate(
            model=model,
            X_test=X_test,
            y_test=y_test,
            label_names=target_names,
        )
        print(f"[{datetime.now()}] Evaluation completed ({len(metrics)} metrics)")

        print(f"[{datetime.now()}] Running cross-validation...")
        t0 = perf_counter()
        cv_metrics = run_cv(
            model_factory(),
            X_train,
            y_train,
        )
        print(
            f"[{datetime.now()}] Cross-validation completed "
            f"({perf_counter() - t0:.1f}s)"
        )

        metrics.update(cv_metrics)

        print(f"[{datetime.now()}] Logging metrics...")
        mlflow.log_metrics(metrics)

        print(f"[{datetime.now()}] Saving classification report...")
        report_path = ARTIFACT_DIR / f"{run_id}_classification_report.json"
        with open(report_path, "w") as f:
            json.dump(
                artifacts["classification_report"],
                f,
                indent=2,
            )
        mlflow.log_artifact(str(report_path))

        print(f"[{datetime.now()}] Saving pitch metrics...")
        f1_pitch_csv_path = ARTIFACT_DIR / f"{run_id}_f1_per_pitch.csv"
        artifacts["f1_per_pitch"].to_csv(
            f1_pitch_csv_path,
            index=False,
        )
        mlflow.log_artifact(str(f1_pitch_csv_path))

        print(f"[{datetime.now()}] Logging confusion matrix...")
        log_confusion_matrix(artifacts["confusion_matrix"])

        print(f"[{datetime.now()}] Logging F1-per-pitch plot...")
        log_f1_per_pitch(artifacts["f1_per_pitch"])

        if artifacts["y_score"] is not None:
            print(f"[{datetime.now()}] Logging precision-recall curve...")
            log_precision_recall_curve(y_test, artifacts["y_score"])

        print(f"[{datetime.now()}] Computing learning curve...")
        t0 = perf_counter()
        log_learning_curve(model_factory(), X_train, y_train)
        print(
            f"[{datetime.now()}] Learning curve completed ({perf_counter() - t0:.1f}s)"
        )

        print(f"[{datetime.now()}] Logging sklearn model...")
        mlflow.sklearn.log_model(model, name="model")
        print(f"[{datetime.now()}] Model logged successfully")

        print("=" * 80)
        print("Run completed")
        print(f"Run ID : {run_id}")

        print("\nMain metrics:")

        summary_metrics = [
            "test_f1_micro",
            "test_f1_macro",
            "test_precision_micro",
            "test_recall_micro",
            "cv_f1_mean",
            "cv_f1_std",
        ]

        for metric in summary_metrics:
            if metric in metrics:
                print(f"{metric}: {metrics[metric]:.4f}")

        print("=" * 80)

In [ ]:
experiments = [
    (
        get_ovr_lr_model,
        "logreg_ovr_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "sklearn",
            "model": "logistic_regression_ovr",
        },
    ),
    (
        get_ovr_svm_model,
        "svm_linear_ovr_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "sklearn",
            "model": "linear_svm_ovr",
        },
    ),
    (
        get_ovr_sgd_model,
        "sgd_logloss_ovr_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "sklearn",
            "model": "sgd_logistic_ovr",
        },
    ),
    (
        get_ovr_hgb_model,
        "hist_gb_ovr_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "sklearn",
            "model": "hist_gradient_boosting_ovr",
        },
    ),
    (
        get_ovr_rf_model,
        "random_forest_ovr_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "sklearn",
            "model": "random_forest_ovr",
        },
    ),
]

for model_factory, run_name, tags in experiments:
    run_experiment(model_factory, run_name, tags)

[2026-06-02 21:12:32.595191] Starting run: logreg_ovr_cqt
[2026-06-02 21:12:32.623186] MLflow run_id: 8a21cd15c21040a0ad1f009f7c39c444
[2026-06-02 21:12:32.678285] Building model...
[2026-06-02 21:12:32.678347] Training model...
[2026-06-02 21:13:57.463429] Training completed (84.8s)
[2026-06-02 21:13:57.463862] Evaluating on test set...
[2026-06-02 21:17:57.927379] Evaluation completed (14 metrics)
[2026-06-02 21:17:57.927722] Running cross-validation...
[2026-06-02 21:19:43.527166] Cross-validation completed (105.6s)
[2026-06-02 21:19:43.527583] Logging metrics...
[2026-06-02 21:19:43.562985] Saving classification report...
[2026-06-02 21:19:43.869251] Saving pitch metrics...
[2026-06-02 21:19:43.925757] Logging confusion matrix...
[2026-06-02 21:19:44.123365] Logging F1-per-pitch plot...
[2026-06-02 21:19:44.314613] Logging precision-recall curve...
[2026-06-02 21:19:45.545242] Computing learning curve...
[2026-06-02 21:29:14.541462] Learning curve completed (569.0s)
[2026-06-02 21:

2026/06/02 21:29:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/02 21:29:17 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-06-02 21:29:17.979445] Model logged successfully
Run completed
Run ID : 8a21cd15c21040a0ad1f009f7c39c444

Main metrics:
cv_f1_mean: 0.5293
cv_f1_std: 0.0137
🏃 View run logreg_ovr_cqt at: http://localhost:5000/#/experiments/1/runs/8a21cd15c21040a0ad1f009f7c39c444
🧪 View experiment at: http://localhost:5000/#/experiments/1
[2026-06-02 21:29:18.051409] Starting run: svm_linear_ovr_cqt
[2026-06-02 21:29:18.112852] MLflow run_id: ec34e36c532043cbaa2b36f6136cfb5a
[2026-06-02 21:29:18.168722] Building model...
[2026-06-02 21:29:18.168779] Training model...
[2026-06-02 21:32:09.615762] Training completed (171.4s)
[2026-06-02 21:32:09.616035] Evaluating on test set...
[2026-06-02 21:34:58.814049] Evaluation completed (14 metrics)
[2026-06-02 21:34:58.814457] Running cross-validation...
[2026-06-02 21:38:26.634954] Cross-validation completed (207.8s)
[2026-06-02 21:38:26.635410] Logging metrics...
[2026-06-02 21:38:26.656732] Saving classification report...
[2026-06-02 21:38:26.709443] Sav

2026/06/02 21:51:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/02 21:51:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-06-02 21:52:00.225853] Model logged successfully
Run completed
Run ID : ec34e36c532043cbaa2b36f6136cfb5a

Main metrics:
cv_f1_mean: 0.5354
cv_f1_std: 0.0142
🏃 View run svm_linear_ovr_cqt at: http://localhost:5000/#/experiments/1/runs/ec34e36c532043cbaa2b36f6136cfb5a
🧪 View experiment at: http://localhost:5000/#/experiments/1
[2026-06-02 21:52:00.295625] Starting run: sgd_logloss_ovr_cqt
[2026-06-02 21:52:00.357114] MLflow run_id: e8c448448cb54ed586b070113a0e8918
[2026-06-02 21:52:00.420791] Building model...
[2026-06-02 21:52:00.420852] Training model...
[2026-06-02 21:55:15.956634] Training completed (195.5s)
[2026-06-02 21:55:15.957038] Evaluating on test set...
[2026-06-02 21:57:46.244454] Evaluation completed (14 metrics)
[2026-06-02 21:57:46.244584] Running cross-validation...
[2026-06-02 22:00:38.386924] Cross-validation completed (172.1s)
[2026-06-02 22:00:38.387491] Logging metrics...
[2026-06-02 22:00:38.438371] Saving classification report...
[2026-06-02 22:00:38.488903

...

2026/06/02 22:41:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/02 22:41:26 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-06-02 22:41:26.586883] Model logged successfully
Run completed
Run ID : b553ae77bc5f4f0bbaa991e683ba899b

Main metrics:
cv_f1_mean: 0.7808
cv_f1_std: 0.0253
🏃 View run hist_gb_ovr_cqt at: http://localhost:5000/#/experiments/1/runs/b553ae77bc5f4f0bbaa991e683ba899b
🧪 View experiment at: http://localhost:5000/#/experiments/1
[2026-06-02 22:41:26.673400] Starting run: random_forest_ovr_cqt
[2026-06-02 22:41:26.739542] MLflow run_id: 5c6dbeb26e064e98a30446e74ee462ec
[2026-06-02 22:41:26.796974] Building model...
[2026-06-02 22:41:26.797650] Training model...
[2026-06-02 23:06:35.288444] Training completed (1508.5s)
[2026-06-02 23:06:35.288585] Evaluating on test set...
[2026-06-02 23:08:46.284011] Evaluation completed (14 metrics)
[2026-06-02 23:08:46.284179] Running cross-validation...
[2026-06-03 00:04:43.847050] Cross-validation completed (3357.6s)
[2026-06-03 00:04:43.847334] Logging metrics...
[2026-06-03 00:04:43.889100] Saving classification report...
[2026-06-03 00:04:43.96132

2026/06/03 02:51:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/03 02:52:22 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-06-03 02:52:49.225135] Model logged successfully
Run completed
Run ID : 5c6dbeb26e064e98a30446e74ee462ec

Main metrics:
cv_f1_mean: 0.7086
cv_f1_std: 0.0490
🏃 View run random_forest_ovr_cqt at: http://localhost:5000/#/experiments/1/runs/5c6dbeb26e064e98a30446e74ee462ec
🧪 View experiment at: http://localhost:5000/#/experiments/1
[2026-06-03 02:52:50.595050] Starting run: knn_ovr_cqt
[2026-06-03 02:52:50.672256] MLflow run_id: a1e124d27ce84758be8294db51b4a32e
[2026-06-03 02:52:50.733659] Building model...
[2026-06-03 02:52:50.735886] Training model...
[2026-06-03 02:52:53.588665] Training completed (2.9s)
[2026-06-03 02:52:53.588884] Evaluating on test set...
[2026-06-03 03:31:01.344693] Evaluation completed (14 metrics)
[2026-06-03 03:31:01.344910] Running cross-validation...
[2026-06-03 04:28:22.088374] Cross-validation completed (3440.8s)
[2026-06-03 04:28:22.088576] Logging metrics...
[2026-06-03 04:28:22.114548] Saving classification report...
[2026-06-03 04:28:22.191573] Savi

KeyboardInterrupt: 

## Ajout d'une PCA

L'analyse du dataset frame-wise disponible dans le notebook [23](./23_eda_dataset_frame_wise.ipynb) montre qu'avec 6 composantes principales, on conversve 92% de variabilité.

In [33]:
N_COMPONENTS_90 = 6
N_COMPONENTS_99 = 35


def get_ovr_pca_model(n_components: int, sub_model: ClassifierMixin) -> Pipeline:
    scaler = RobustScaler()
    pca = PCA(n_components=n_components)

    model = OneVsRestClassifier(sub_model)

    return Pipeline(
        steps=[
            ("scaler", scaler),
            ("pca", pca),
            ("model", model),
        ]
    )


get_ovr_lr_pca90_model = functools.partial(
    get_ovr_pca_model,
    n_components=N_COMPONENTS_90,
    sub_model=LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1,
    ),
)

get_ovr_lr_pca99_model = functools.partial(
    get_ovr_pca_model,
    n_components=N_COMPONENTS_99,
    sub_model=LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1,
    ),
)

In [ ]:
experiments_pca = [
    (
        get_ovr_lr_pca90_model,
        "pca90_logreg_ovr_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "sklearn",
            "model": "pca90_logistic_regression_ovr",
        },
    ),
    (
        get_ovr_lr_pca99_model,
        "pca99_logreg_ovr_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "sklearn",
            "model": "pca99_logistic_regression_ovr",
        },
    ),
]

for model_factory, run_name, tags in experiments_pca:
    run_experiment(model_factory, run_name, tags)

[2026-06-05 16:43:16.381954] Starting run: pca6_logreg_ovr_cqt
[2026-06-05 16:43:16.423578] MLflow run_id: a9718c6b47a0412bbe4fd42b7c455fab
[2026-06-05 16:43:16.490347] Building model...
[2026-06-05 16:43:16.490407] Training model...
[2026-06-05 16:43:26.487662] Training completed (10.0s)
[2026-06-05 16:43:26.487972] Evaluating on test set...
[2026-06-05 16:59:51.000792] Evaluation completed (14 metrics)
[2026-06-05 16:59:51.001215] Running cross-validation...
[2026-06-05 17:00:13.019258] Cross-validation completed (22.0s)
[2026-06-05 17:00:13.019545] Logging metrics...
[2026-06-05 17:00:13.076835] Saving classification report...
[2026-06-05 17:00:13.129896] Saving pitch metrics...
[2026-06-05 17:00:13.182216] Logging confusion matrix...
[2026-06-05 17:00:13.463876] Logging F1-per-pitch plot...
[2026-06-05 17:00:13.796465] Logging precision-recall curve...
[2026-06-05 17:00:15.714487] Computing learning curve...
[2026-06-05 17:01:14.883824] Learning curve completed (59.2s)
[2026-06-05 

2026/06/05 17:01:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/05 17:01:17 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-06-05 17:01:17.993396] Model logged successfully
Run completed
Run ID : a9718c6b47a0412bbe4fd42b7c455fab

Main metrics:
cv_f1_mean: 0.1650
cv_f1_std: 0.0108
🏃 View run pca6_logreg_ovr_cqt at: http://localhost:5000/#/experiments/1/runs/a9718c6b47a0412bbe4fd42b7c455fab
🧪 View experiment at: http://localhost:5000/#/experiments/1
[2026-06-05 17:01:18.064148] Starting run: pca6_logreg_ovr_cqt
[2026-06-05 17:01:18.127738] MLflow run_id: 8f55afd5d20448fa95e916f22c3443f9
[2026-06-05 17:01:18.182966] Building model...
[2026-06-05 17:01:18.183013] Training model...
[2026-06-05 17:02:02.742494] Training completed (44.6s)
[2026-06-05 17:02:02.742641] Evaluating on test set...
[2026-06-05 17:05:39.173221] Evaluation completed (14 metrics)
[2026-06-05 17:05:39.173654] Running cross-validation...
[2026-06-05 17:07:07.333183] Cross-validation completed (88.2s)
[2026-06-05 17:07:07.333859] Logging metrics...
[2026-06-05 17:07:07.358482] Saving classification report...
[2026-06-05 17:07:07.409892]

2026/06/05 17:13:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/05 17:13:05 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-06-05 17:13:05.775817] Model logged successfully
Run completed
Run ID : 8f55afd5d20448fa95e916f22c3443f9

Main metrics:
cv_f1_mean: 0.4323
cv_f1_std: 0.0109
🏃 View run pca6_logreg_ovr_cqt at: http://localhost:5000/#/experiments/1/runs/8f55afd5d20448fa95e916f22c3443f9
🧪 View experiment at: http://localhost:5000/#/experiments/1


## Optimisation 

In [ ]:
N_ITERATION = 100

def get_ovr_hgb_search() -> RandomizedSearchCV:
    scaler = RobustScaler()

    model = OneVsRestClassifier(
        HistGradientBoostingClassifier(
            random_state=RANDOM_STATE,
        )
    )

    pipeline = Pipeline(
        steps=[
            ("scaler", scaler),
            ("model", model),
        ]
    )

    param_distributions = {
        "model__estimator__learning_rate": loguniform(1e-3, 3e-1),
        "model__estimator__max_depth": randint(3, 15),
        "model__estimator__max_iter": randint(50, 500),
        "model__estimator__min_samples_leaf": randint(10, 200),
        "model__estimator__l2_regularization": loguniform(1e-8, 1.0),
        "model__estimator__max_leaf_nodes": randint(15, 255),
    }

    return RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=N_ITERATION,
        scoring="f1_micro",
        cv=5,
        n_jobs=-1,
        verbose=2,
        random_state=RANDOM_STATE,
        refit=True,
    )

In [ ]:
def run_optimization(search_factory: callable, run_name: str, tags: dict) -> dict:

    print("=" * 80)
    print(f"[{datetime.now()}] Starting run: {run_name}")

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id
        print(f"[{datetime.now()}] MLflow run_id: {run_id}")

        mlflow.set_tags(tags)

        print(f"[{datetime.now()}] Building search...")
        search = search_factory()

        print(f"[{datetime.now()}] Training search...")
        t0 = perf_counter()
        search.fit(X_train, y_train)
        print(f"[{datetime.now()}] Training completed ({perf_counter() - t0:.1f}s)")

        print(f"[{datetime.now()}] Logging best score...")
        mlflow.log_metric("best_cv_score", search.best_score_)

        print(f"[{datetime.now()}] Logging best params...")
        mlflow.log_params(
            {f"best_{k}": v for k, v in search.best_params_.items()}
        )

        print(f"[{datetime.now()}] Logging trials...")
        for run_idx, params in enumerate(search.cv_results_["params"], 1):
            with mlflow.start_run(run_name=f"trial_{run_idx}", nested=True):
                mlflow.log_params(params)
                mlflow.log_metric(
                    "mean_cv_score",
                    search.cv_results_["mean_test_score"][run_idx - 1],
                )
                mlflow.log_metric(
                    "std_cv_score",
                    search.cv_results_["std_test_score"][run_idx - 1],
                )

        print(f"[{datetime.now()}] Logging cv_results...")
        cv_results = pd.DataFrame(search.cv_results_)
        cv_results_path = (
            ARTIFACT_DIR / f"{run_id}_cv_results.csv"
        )
        cv_results.to_csv(
            cv_results_path,
            index=False,
        )
        mlflow.log_artifact(str(cv_results_path))

        print("=" * 80)
        print("Run completed")
        print(f"Run ID : {run_id}")

        print("\nBest Score:")
        best_idx = search.best_index_
        print(
            f"f1_micro: {search.cv_results_["mean_test_score"][best_idx]} "
            f"(+/- {search.cv_results_["std_test_score"][best_idx]})"
        )

        print("\nBest params:")
        best_params = search.best_params_
        for key, value in best_params.items():
            print(f"{key}: {value}")

        print("=" * 80)

        return best_params

In [ ]:
best_params = run_optimization(
    search_factory=get_ovr_hgb_search,
    run_name="ovr_hgb_random_search",
    tags={
        "task": "audio_to_midi",
        "representation": "cqt",
        "model_family": "sklearn",
        "model": "hist_gradient_boosting_ovr",
    }
)

In [ ]:
def get_best_ovr_hgb_model():
    scaler = RobustScaler()

    model = OneVsRestClassifier(
        HistGradientBoostingClassifier(
            learning_rate=best_params["model__estimator__learning_rate"],
            max_depth=best_params["model__estimator__max_depth"],
            max_iter=best_params["model__estimator__max_iter"],
            min_samples_leaf=best_params["model__estimator__min_samples_leaf"],
            l2_regularization=best_params["model__estimator__l2_regularization"],
            max_leaf_nodes=best_params["model__estimator__max_leaf_nodes"],
            random_state=RANDOM_STATE,
        )
    )

    return Pipeline(
        steps=[
            ("scaler", scaler),
            ("model", model),
        ]
    )

In [ ]:
run_experiment(
    model_factory=get_best_ovr_hgb_model,
    run_name="best_ovr_hgb_model",
    tags={
        "task": "audio_to_midi",
        "representation": "cqt",
        "model_family": "sklearn",
        "model": "hist_gradient_boosting_ovr",
    }
)